# Train an SO101-Nexus policy with PPO on MuJoCo Warp

Self-contained [SO101-Nexus](https://so101-nexus.com/docs) demo: large-batch **PPO** on the
GPU-parallel **MuJoCo Warp** backend. It steps thousands of worlds in place on the GPU, so the
whole PPO hot path stays on-device. The solved-baseline notebook path covers PickLift,
touch, look-at, and move; pick your environment in step 3.

On one RTX 5090, `WarpPickLift-v1` reached 0.993 final recent-episode success and
1.000 best recent success in 31.6 minutes; touch, look-at, and move solve in about a
minute. Colab A100/L4 are slower; a free T4 works but is slow (lower `--num-envs`).

Metrics stream to **TensorBoard** (embedded below): watch `charts/success_rate` climb live.


## 1. Check the GPU

Runtime -> Change runtime type -> GPU. Warp needs a CUDA device.


In [ ]:
!nvidia-smi -L

## 2. Install SO101-Nexus (Warp + training extras)

Clone the repo (the example scripts live in `examples/`, not the wheel) and install the
`warp` + `train` extras. Colab ships a CUDA build of PyTorch already.


In [ ]:
!git clone --depth 1 https://github.com/johnsutor/so101-nexus.git
%cd so101-nexus
!pip install -q -e ".[warp,train]" "imageio[ffmpeg]"

## 3. Choose the environment

Set `ENV_ID` to the task you want to train. PickLift needs the most steps; touch,
look-at, and move solve fast. Pick-and-place is excluded from the solved baselines
until the environment is fixed.


In [ ]:
# Options: WarpPickLift-v1, WarpTouch-v1, WarpLookAt-v1, WarpMove-v1
ENV_ID = "WarpPickLift-v1"
STEPS = {"WarpPickLift-v1": 100_000_000}.get(ENV_ID, 5_000_000)
print(f"Training {ENV_ID} for {STEPS:,} env steps")

## 4. Launch TensorBoard

Run this before training so the dashboard picks up the new run and refreshes live.


In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

## 5. Train

Defaults are the winning recipe (fixed-horizon episodes, entropy annealed 0.01 -> 0,
staggered resets, obs/reward normalization) with 1024 GPU-parallel worlds. The
entropy and learning-rate schedules run over `--total-timesteps`, so the step budget
is part of the recipe. `best_agent.pt` (policy + obs-norm stats) is saved to
`runs/...` on every success-rate improvement. Reduce `--num-envs` (e.g. 512 or 256)
on smaller GPUs.


In [ ]:
!python examples/ppo_warp.py --exp-name colab --env-id {ENV_ID} --total-timesteps {STEPS}

## 6. Evaluate the trained policy (deterministic success rate)

Runs the saved policy with no exploration noise across 512 worlds and reports the fraction of
episodes that solve the task (`success_rate(ever)`), the hold rate at the final step, and mean
return.


In [ ]:
CKPT = f"runs/{ENV_ID}__colab__*/best_agent.pt"
!python examples/eval_warp.py --env-id {ENV_ID} --checkpoint "{CKPT}"

## 7. Watch a rollout

Training periodically renders a deterministic eval in the matching `MuJoCo*` backend (a
transfer figure, since Warp uses a slightly different integrator). The newest clip:


In [ ]:
import base64
import glob

from IPython.display import HTML

clips = sorted(glob.glob(f"runs/{ENV_ID}__colab__*/videos/*.mp4"))
assert clips, "no eval video yet; let training reach its first --eval-freq checkpoint"
with open(clips[-1], "rb") as f:
    data = base64.b64encode(f.read()).decode()
src = f"data:video/mp4;base64,{data}"
HTML(f'<video controls autoplay loop width=480><source src="{src}" type="video/mp4"></video>')

## Next steps

- **Reproducibility.** Lift discovery on PickLift is seed-sensitive; if a run stalls at
  low return / 0% success, try another `--seed` (`--stagger-resets` is on by default).
- **Tuning.** See [`examples/README.md`](https://github.com/johnsutor/so101-nexus/blob/main/examples/README.md)
  and [Training with PPO](https://so101-nexus.com/docs/guides/training-with-ppo).
